# Continuous Difference-in-Differences

**Estimating dose-response effects with heterogeneous treatment intensity**

*Based on Callaway, Goodman-Bacon & Sant'Anna (2024)*

---

Standard difference-in-differences compares treated and untreated groups. But what happens when treatment isn't binary? In a **job training program**, some workers receive 5 hours of training, others get 20 hours, and some get 50. Binary DiD lumps all trained workers together, losing the rich variation in treatment intensity. **Continuous DiD** estimates how earnings respond to each level of training hours — a full dose-response curve.

### When to use `ContinuousDiD`

- Treatment has **continuous intensity or dose** (hours, dollars, distance, etc.)
- You want a **dose-response curve** showing how effects vary with dose
- You need **marginal effects** (what does one more unit of treatment do?)
- You want to **avoid binarizing** a naturally continuous treatment
- An **untreated comparison group** exists (never-treated or not-yet-treated units)

### Topics covered

1. Why continuous treatment needs special methods
2. Data setup and the dose distribution
3. Basic estimation with `ContinuousDiD`
4. Dose-response curves: ATT(d) and ACRT(d)
5. Event study diagnostics
6. Advanced features: splines, control groups, bootstrap
7. Comparison to binary DiD (information loss from binarizing)
8. Results export and group-time effects

**Prerequisites:** Familiarity with standard DiD ([Tutorial 02: Staggered DiD](02_staggered_did.ipynb)) and parallel trends ([Tutorial 04](04_parallel_trends.ipynb)).

In [ ]:
import numpy as np
import pandas as pd
from diff_diff import ContinuousDiD, CallawaySantAnna, generate_continuous_did_data

try:
    import matplotlib.pyplot as plt
    plt.style.use('seaborn-v0_8-whitegrid')
    HAS_MATPLOTLIB = True
except ImportError:
    HAS_MATPLOTLIB = False
    print("matplotlib not installed - visualization examples will be skipped")

## 1. Why Continuous Treatment Needs Special Methods

Consider a job training program. Workers receive different amounts of training, and we want to know: **does more training lead to higher earnings?** Three problems arise with naive approaches:

**Problem 1: Binarizing loses information.** Collapsing 5 hours vs. 20 hours vs. 50 hours into "trained / untrained" tells you whether *any* training helps, but not whether *more* training helps more. You lose the dose-response relationship entirely.

**Problem 2: TWFE with continuous treatment is biased.** Plugging a continuous treatment variable into a standard two-way fixed effects regression can produce biased estimates due to negative weights, contamination bias, and scale dependence — the same issues that plague binary TWFE in staggered settings, but compounded.

**Problem 3: You need new estimands.** With continuous treatment, we need to distinguish:

| Parameter | Meaning (training example) |
|-----------|---------------------------|
| ATT(d) | Total earnings gain from *d* hours of training vs. no training, across all treated workers (requires strong PT) |
| ACRT(d) | Earnings gain from **one additional hour** at level *d* (marginal effect; requires strong PT) |
| ATT_glob | Average effect of any training vs. none (binarized). Identifies ATT_loc under standard PT; equals ATT_glob under strong PT |
| ACRT_glob | Average marginal return to an extra hour of training (requires strong PT) |

The `ContinuousDiD` estimator from Callaway, Goodman-Bacon & Sant'Anna (2024) handles all three problems. It estimates the full dose-response curve using B-splines and provides valid inference.

### Identifying assumptions

The continuous DiD framework relies on two levels of parallel trends:

- **Standard parallel trends:** Untreated potential outcomes evolve identically regardless of treatment dose. This is the same assumption as binary DiD, extended to all dose levels. Under standard PT, the estimator identifies the effect of dose *d* on units who actually received dose *d* — and the local average of that effect across dose groups. However, you cannot compare effects across different dose levels or give a causal interpretation to the dose-response curve.

- **Strong parallel trends:** Additionally, there is no selection into dose based on potential gains. That is, workers who chose 50 hours of training would have had the same earnings trajectory as workers who chose 5 hours, had they both remained untreated. Under strong PT, the full **dose-response curve** ATT(d) is identified, cross-dose comparisons are valid, and all ACRT parameters (both dose-specific and global) have a causal interpretation.

Strong parallel trends is needed for the dose-response curve and all marginal effect (ACRT) parameters; standard PT only identifies the binarized overall effect and its local average.

## 2. Data Setup

We generate a balanced panel of workers. Some are assigned to a training cohort (treated in period 2), while 30% are never treated. Each trained worker receives a dose (hours) drawn from a log-normal distribution. The true treatment effect is **ATT(d) = 1 + 2d** — each hour of training adds \$2 to earnings, plus a base effect of \$1 for any training at all.

In [ ]:
data = generate_continuous_did_data(
    n_units=500,
    n_periods=4,
    cohort_periods=[2],
    never_treated_frac=0.3,
    att_function="linear",
    att_intercept=1.0,
    att_slope=2.0,
    seed=42,
)

print(f"Shape: {data.shape}")
print(f"Units: {data['unit'].nunique()}, Periods: {data['period'].nunique()}")
print(f"Treatment cohorts: {sorted(data.loc[data['first_treat'] > 0, 'first_treat'].unique())}")
print(f"Never-treated units: {(data.groupby('unit')['first_treat'].first() == 0).sum()}")
print()
data.head(10)

In [ ]:
# Dose distribution for treated units
treated_doses = data.loc[data['first_treat'] > 0].groupby('unit')['dose'].first()

print("Dose summary (treated units only):")
print(f"  Min:    {treated_doses.min():.2f}")
print(f"  Max:    {treated_doses.max():.2f}")
print(f"  Mean:   {treated_doses.mean():.2f}")
print(f"  Median: {treated_doses.median():.2f}")

if HAS_MATPLOTLIB:
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))

    # Left: dose histogram
    axes[0].hist(treated_doses, bins=30, edgecolor='white', alpha=0.8, color='steelblue')
    axes[0].set_xlabel('Training hours (dose)')
    axes[0].set_ylabel('Number of workers')
    axes[0].set_title('Distribution of Training Hours')

    # Right: true ATT(d) curve
    d_grid = np.linspace(treated_doses.min(), treated_doses.max(), 100)
    axes[1].plot(d_grid, 1.0 + 2.0 * d_grid, 'k-', linewidth=2)
    axes[1].set_xlabel('Training hours (dose)')
    axes[1].set_ylabel('True ATT(d)')
    axes[1].set_title('True Dose-Response: ATT(d) = 1 + 2d')

    plt.tight_layout()
    plt.show()

## 3. Basic Estimation

The `ContinuousDiD` estimator follows an sklearn-like API. Call `fit()` with column names for outcome, unit, time, first treatment period, and dose. Setting `aggregate="dose"` computes the dose-response curve and global summary parameters.

In [ ]:
est = ContinuousDiD(seed=42)
results = est.fit(
    data,
    outcome="outcome",
    unit="unit",
    time="period",
    first_treat="first_treat",
    dose="dose",
    aggregate="dose",
)

print(results.summary())

In [ ]:
# Compare estimated vs. true parameters
true_att_glob = 1.0 + 2.0 * treated_doses.mean()
true_acrt_glob = 2.0

print("Parameter comparison:")
print(f"  ATT_glob  — estimated: {results.overall_att:.4f}, true: {true_att_glob:.4f}, "
      f"bias: {results.overall_att - true_att_glob:.4f}")
print(f"  ACRT_glob — estimated: {results.overall_acrt:.4f}, true: {true_acrt_glob:.4f}, "
      f"bias: {results.overall_acrt - true_acrt_glob:.4f}")

### Interpreting the global parameters

- **ATT_glob** is the average total effect of training (at the observed dose levels) compared to no training. It answers: "On average, how much more do trained workers earn?" The API reports this as `overall_att`. Under standard PT, this identifies the local average effect (ATT_loc); under strong PT, it additionally equals the global average (ATT_glob).
- **ACRT_glob** is the average marginal return to an additional hour of training. It answers: "What is one more hour of training worth, on average?" This requires strong PT for causal interpretation.

Analytical standard errors are the default (fast, no resampling). For bootstrap-based inference, set `n_bootstrap` (see Section 6).

## 4. Dose-Response Curves

The key advantage of continuous DiD is the **dose-response curve** — how the treatment effect varies across dose levels. The estimator evaluates ATT(d) and ACRT(d) on a grid of dose values (by default, percentiles from P10 to P99 of the observed dose distribution).

Access the curves via `results.dose_response_att` and `results.dose_response_acrt`, which are `DoseResponseCurve` objects with attributes `dose_grid`, `effects`, `se`, `conf_int_lower`, and `conf_int_upper`.

In [ ]:
# Tabular view of the ATT dose-response curve
att_df = results.dose_response_att.to_dataframe()
print("ATT(d) dose-response curve:")
print(att_df.head(10).to_string(index=False))

In [ ]:
if HAS_MATPLOTLIB:
    fig, axes = plt.subplots(1, 2, figsize=(13, 5))

    # ATT(d)
    att = results.dose_response_att
    axes[0].plot(att.dose_grid, att.effects, 'b-', linewidth=2, label='Estimated ATT(d)')
    axes[0].fill_between(att.dose_grid, att.conf_int_lower, att.conf_int_upper,
                         alpha=0.2, color='blue')
    axes[0].plot(att.dose_grid, 1.0 + 2.0 * att.dose_grid, 'k--', linewidth=1.5,
                 label='True ATT(d)')
    axes[0].set_xlabel('Training hours (dose)')
    axes[0].set_ylabel('ATT(d)')
    axes[0].set_title('Dose-Response: Total Effect')
    axes[0].legend()

    # ACRT(d)
    acrt = results.dose_response_acrt
    axes[1].plot(acrt.dose_grid, acrt.effects, 'r-', linewidth=2, label='Estimated ACRT(d)')
    axes[1].fill_between(acrt.dose_grid, acrt.conf_int_lower, acrt.conf_int_upper,
                         alpha=0.2, color='red')
    axes[1].axhline(2.0, color='k', linestyle='--', linewidth=1.5, label='True ACRT = 2.0')
    axes[1].set_xlabel('Training hours (dose)')
    axes[1].set_ylabel('ACRT(d)')
    axes[1].set_title('Dose-Response: Marginal Effect')
    axes[1].legend()

    plt.tight_layout()
    plt.show()

### Interpreting the dose-response curves

- **Left panel (ATT):** The total effect of training rises roughly linearly with dose, closely tracking the true curve (dashed). A worker with 3 hours of training gains about $7 in earnings; a worker with 5 hours gains about $11.
- **Right panel (ACRT):** The marginal return to one additional hour is approximately constant at $2, matching the true DGP. The confidence band is wider at extreme doses where fewer workers are observed.

These curves require the **strong parallel trends** assumption. Under standard PT only, the overall binarized effect is identified — this is ATT_loc (the local average across dose groups). The dose-response curve and ACRT_glob are not identified under standard PT because they involve cross-dose comparisons and counterfactual dose-response derivatives. The ATT_glob reported by the estimator is mechanically the binarized DiD: under standard PT it equals ATT_loc, while under strong PT it additionally equals the global average ATT_glob.

## 5. Event Study Diagnostics

An event study aggregation binarizes treatment (treated vs. untreated) and shows effects by relative time period. This is useful for **pre-trends diagnostics**: pre-treatment coefficients should be near zero if parallel trends holds.

Event study works best with multiple cohorts (to have richer variation in event-time). A single cohort can still produce event-time estimates, but diagnostics are more informative with multiple cohorts. We generate a new dataset with two treatment cohorts.

In [ ]:
# Generate multi-cohort data for event study
data_es = generate_continuous_did_data(
    n_units=500,
    n_periods=6,
    cohort_periods=[3, 5],
    never_treated_frac=0.3,
    att_function="linear",
    att_intercept=1.0,
    att_slope=2.0,
    seed=42,
)

est_es = ContinuousDiD(n_bootstrap=199, seed=42)
results_es = est_es.fit(
    data_es,
    outcome="outcome",
    unit="unit",
    time="period",
    first_treat="first_treat",
    dose="dose",
    aggregate="eventstudy",
)

# Event study table
es_df = results_es.to_dataframe(level="event_study")
es_df['pre_period'] = es_df['relative_period'] < 0
print("Event Study Effects (binarized ATT by relative period):")
print(es_df.to_string(index=False))

In [ ]:
if HAS_MATPLOTLIB:
    fig, ax = plt.subplots(figsize=(8, 5))

    rel = es_df['relative_period'].values
    att = es_df['att_glob'].values
    ci_lo = es_df['conf_int_lower'].values
    ci_hi = es_df['conf_int_upper'].values

    ax.errorbar(rel, att, yerr=[att - ci_lo, ci_hi - att],
                fmt='o', capsize=4, color='steelblue', markersize=6, linewidth=1.5)
    ax.axhline(0, color='gray', linestyle='-', linewidth=0.8)
    ax.axvline(-0.5, color='gray', linestyle='--', linewidth=0.8, label='Treatment onset')
    ax.set_xlabel('Periods relative to treatment')
    ax.set_ylabel('ATT (binarized)')
    ax.set_title('Event Study: Pre-Trends Diagnostic')
    ax.legend()

    plt.tight_layout()
    plt.show()

## 6. Advanced Features

### B-spline configuration

The estimator approximates the dose-response function using B-splines. Two parameters control the flexibility:

- **`degree`** controls smoothness: `degree=1` gives a piecewise-linear fit, `degree=3` (default) gives a smooth cubic curve.
- **`num_knots`** adds interior knots for additional flexibility. With 0 knots (default), the spline is a single polynomial segment. Adding knots lets the curve bend at more points, useful for non-linear dose-response relationships.

Let's see how these settings affect estimation on data with a **log dose-response**: ATT(d) = 1 + 2 log(1 + d).

In [ ]:
# Generate data with log dose-response
data_log = generate_continuous_did_data(
    n_units=500,
    n_periods=4,
    cohort_periods=[2],
    never_treated_frac=0.3,
    att_function="log",
    att_intercept=1.0,
    att_slope=2.0,
    seed=42,
)

# Compare three B-spline configurations
configs = [
    {"label": "Linear (degree=1, knots=0)", "degree": 1, "num_knots": 0},
    {"label": "Cubic (degree=3, knots=0)", "degree": 3, "num_knots": 0},
    {"label": "Cubic + knots (degree=3, knots=2)", "degree": 3, "num_knots": 2},
]

spline_results = {}
print(f"{'Configuration':<35} {'ATT_glob':>10} {'ACRT_glob':>11}")
print("-" * 58)
for cfg in configs:
    est_cfg = ContinuousDiD(degree=cfg["degree"], num_knots=cfg["num_knots"], seed=42)
    res_cfg = est_cfg.fit(data_log, "outcome", "unit", "period", "first_treat", "dose",
                          aggregate="dose")
    spline_results[cfg["label"]] = res_cfg
    print(f"{cfg['label']:<35} {res_cfg.overall_att:>10.4f} {res_cfg.overall_acrt:>11.4f}")

In [ ]:
if HAS_MATPLOTLIB:
    fig, ax = plt.subplots(figsize=(8, 5))

    colors = ['#e74c3c', '#3498db', '#2ecc71']
    for (label, res), color in zip(spline_results.items(), colors):
        att = res.dose_response_att
        ax.plot(att.dose_grid, att.effects, color=color, linewidth=2, label=label)

    # True curve
    log_doses = data_log.loc[data_log['first_treat'] > 0].groupby('unit')['dose'].first()
    d_grid = np.linspace(log_doses.min(), log_doses.max(), 100)
    ax.plot(d_grid, 1.0 + 2.0 * np.log1p(d_grid), 'k--', linewidth=2, label='True: 1 + 2 log(1+d)')

    ax.set_xlabel('Training hours (dose)')
    ax.set_ylabel('ATT(d)')
    ax.set_title('B-Spline Configuration: Effect on Dose-Response Fit')
    ax.legend(fontsize=9)

    plt.tight_layout()
    plt.show()

### Control group options

The `control_group` parameter selects which untreated units serve as the comparison group:

- **`"never_treated"`** (default): Only units that are never treated. Stronger assumption (parallel trends must hold for all periods) but avoids contamination from units that are treated later.
- **`"not_yet_treated"`**: Units not yet treated at time *t*. Weaker assumption and more data, but treated-later units may already be adjusting behavior in anticipation.

In [ ]:
# Multi-cohort data for control group comparison
data_mc = generate_continuous_did_data(
    n_units=500,
    n_periods=6,
    cohort_periods=[3, 5],
    never_treated_frac=0.3,
    att_function="linear",
    att_intercept=1.0,
    att_slope=2.0,
    seed=42,
)

print(f"{'Control group':<20} {'ATT_glob':>10} {'SE':>10} {'ACRT_glob':>11} {'SE':>10}")
print("-" * 63)
for cg in ["never_treated", "not_yet_treated"]:
    est_cg = ContinuousDiD(control_group=cg, seed=42)
    res_cg = est_cg.fit(data_mc, "outcome", "unit", "period", "first_treat", "dose",
                        aggregate="dose")
    print(f"{cg:<20} {res_cg.overall_att:>10.4f} {res_cg.overall_att_se:>10.4f} "
          f"{res_cg.overall_acrt:>11.4f} {res_cg.overall_acrt_se:>10.4f}")

### Bootstrap inference

By default, `ContinuousDiD` computes **analytical standard errors** (fast, no resampling). For more robust inference, enable the **multiplier bootstrap** by setting `n_bootstrap`. The bootstrap perturbs unit-level influence functions using random weights.

Weight options:
- `"rademacher"` (default): ±1 with equal probability. Good for most cases.
- `"mammen"`: Two-point distribution matching first 3 moments.
- `"webb"`: Six-point distribution, recommended for very few clusters (<10).

We recommend `n_bootstrap >= 199` for reliable inference.

In [ ]:
# Analytical SEs (default)
est_ana = ContinuousDiD(seed=42)
res_ana = est_ana.fit(data, "outcome", "unit", "period", "first_treat", "dose",
                      aggregate="dose")

# Bootstrap SEs
est_boot = ContinuousDiD(n_bootstrap=199, seed=42)
res_boot = est_boot.fit(data, "outcome", "unit", "period", "first_treat", "dose",
                        aggregate="dose")

print(f"{'Method':<15} {'ATT_glob':>10} {'SE':>10} {'95% CI':>25}")
print("-" * 62)
for label, r in [("Analytical", res_ana), ("Bootstrap", res_boot)]:
    ci = r.overall_att_conf_int
    print(f"{label:<15} {r.overall_att:>10.4f} {r.overall_att_se:>10.4f} "
          f"[{ci[0]:>8.4f}, {ci[1]:>8.4f}]")

print()
print(f"{'Method':<15} {'ACRT_glob':>10} {'SE':>10} {'95% CI':>25}")
print("-" * 62)
for label, r in [("Analytical", res_ana), ("Bootstrap", res_boot)]:
    ci = r.overall_acrt_conf_int
    print(f"{label:<15} {r.overall_acrt:>10.4f} {r.overall_acrt_se:>10.4f} "
          f"[{ci[0]:>8.4f}, {ci[1]:>8.4f}]")

## 7. Comparison to Binary DiD

What if we ignore dose entirely and just run a standard binary Callaway-Sant'Anna estimator? Both approaches should give a similar **binarized ATT** (treated vs. untreated), but the binary approach discards all dose information — no dose-response curve, no marginal effects.

Note: both estimators compute the binarized ATT as a simple mean difference (treated vs. untreated), so the values should be very close. Under standard PT this identifies ATT_loc (the local average); under strong PT it additionally equals ATT_glob. Any small differences arise from weighting or aggregation choices, control group or base period settings, or finite-sample variation — not from spline smoothing. The continuous approach provides the full dose-response curve on top of the binarized effect.

In [ ]:
# Binary DiD: Callaway-Sant'Anna (ignores dose)
cs = CallawaySantAnna(control_group="never_treated")
results_binary = cs.fit(
    data,
    outcome="outcome",
    unit="unit",
    time="period",
    first_treat="first_treat",
)

print(f"{'Estimator':<25} {'Overall ATT':>12} {'SE':>10}")
print("-" * 49)
print(f"{'ContinuousDiD':<25} {results.overall_att:>12.4f} {results.overall_att_se:>10.4f}")
print(f"{'CallawaySantAnna':<25} {results_binary.overall_att:>12.4f} {results_binary.overall_se:>10.4f}")
print()
print("Binary DiD gives one number. Continuous DiD gives the full dose-response curve")
print("plus marginal effects (ACRT) — without losing any information.")

In [ ]:
if HAS_MATPLOTLIB:
    fig, ax = plt.subplots(figsize=(9, 5))

    # Dose-response curve with CI
    att = results.dose_response_att
    ax.plot(att.dose_grid, att.effects, 'b-', linewidth=2, label='Continuous DiD: ATT(d)')
    ax.fill_between(att.dose_grid, att.conf_int_lower, att.conf_int_upper,
                    alpha=0.15, color='blue')

    # Binary ATT (flat line)
    ax.axhline(results_binary.overall_att, color='red', linestyle='--', linewidth=2,
               label=f'Binary DiD: ATT = {results_binary.overall_att:.2f}')

    # True curve
    ax.plot(att.dose_grid, 1.0 + 2.0 * att.dose_grid, 'k:', linewidth=2,
            label='True: ATT(d) = 1 + 2d')

    ax.set_xlabel('Training hours (dose)')
    ax.set_ylabel('Effect on earnings')
    ax.set_title('Continuous vs. Binary DiD: Information Loss from Binarizing')
    ax.legend()

    plt.tight_layout()
    plt.show()

    print("The red dashed line (binary DiD) collapses the entire dose-response curve")
    print("into a single number, losing the relationship between dose and effect.")

## 8. Results Export and Group-Time Effects

Results can be exported to DataFrames at different levels of aggregation using `to_dataframe()`:

- **`level="dose_response"`** (default): The dose-response curve with ATT(d) and ACRT(d)
- **`level="group_time"`**: Underlying group-time cell estimates
- **`level="event_study"`**: Event study effects (only available when fitted with `aggregate="eventstudy"`)

In [ ]:
# Dose-response level
print("Dose-response level:")
dr_df = results.to_dataframe(level="dose_response")
print(dr_df.head().to_string(index=False))

print()

# Group-time level
print("Group-time level:")
gt_df = results.to_dataframe(level="group_time")
print(gt_df.to_string(index=False))

## 9. Summary

### Key takeaways

- **Continuous DiD** estimates dose-response effects without binarizing treatment intensity
- **ATT(d)** gives the total effect at dose *d*; **ACRT(d)** gives the marginal effect
- **ATT_glob** and **ACRT_glob** summarize the overall and marginal effects
- **B-splines** approximate the dose-response flexibly; increase `degree` and `num_knots` for non-linear patterns
- **Event study** diagnostics work the same as in binary DiD (pre-period coefficients near zero)
- Binarizing continuous treatment into binary DiD **loses information** — continuous DiD recovers the full curve

### Parameter reference

| Parameter | Default | Description |
|-----------|---------|-------------|
| `degree` | 3 | B-spline polynomial degree (1 = linear, 3 = cubic) |
| `num_knots` | 0 | Interior knots for spline flexibility |
| `control_group` | `"never_treated"` | `"never_treated"` or `"not_yet_treated"` |
| `anticipation` | 0 | Periods of treatment anticipation |
| `base_period` | `"varying"` | `"varying"` or `"universal"` |
| `n_bootstrap` | 0 | Bootstrap iterations (0 = analytical SEs only) |
| `bootstrap_weights` | `"rademacher"` | `"rademacher"`, `"mammen"`, or `"webb"` |
| `seed` | `None` | Random seed for reproducibility |
| `aggregate` | `None` | `"dose"` for dose-response, `"eventstudy"` for event study |

### Reference

Callaway, B., Goodman-Bacon, A., & Sant'Anna, P. H. C. (2024). Difference-in-Differences with a Continuous Treatment. [arXiv:2107.02637](https://arxiv.org/abs/2107.02637).